# Module 3: Temporal Difference Learning
## Notebook 1: TD(0) Prediction

**Learning Objectives**

By completing this notebook, you will:
1. Implement TD(0) prediction from scratch using the Bellman bootstrapping update
2. Apply TD(0) to the Random Walk environment and understand why it works
3. Compare TD(0) convergence speed against Monte Carlo prediction on the same task
4. Visualize learned value estimates versus analytically-computed true values

**Prerequisites**
- Monte Carlo prediction (Module 2)
- The Bellman expectation equation
- Python, NumPy, Matplotlib

**Estimated Time: 14 minutes**

---

### Why TD Learning?

Monte Carlo methods wait until an episode ends before updating values. This is slow and impossible in continuing (non-episodic) tasks. **Temporal Difference (TD) learning** bootstraps: it updates estimates using other estimates before the episode ends.

The core TD(0) update is:

$$V(S_t) \leftarrow V(S_t) + \alpha \left[ R_{t+1} + \gamma V(S_{t+1}) - V(S_t) \right]$$

The quantity $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ is called the **TD error** — the difference between what we expected and what we got (plus the updated estimate of future value).

In [ ]:
# Purpose: Import all dependencies and set reproducibility seeds
# Key Concept: Every experiment in this notebook is reproducible

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)

# Display settings
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print("Setup complete.")

## 1. The Random Walk Environment

We use the classic **7-state Random Walk** (Sutton & Barto, Example 6.2). States are labeled A through G. The agent starts at the center state D and moves left or right with equal probability at each step. Reaching state A (leftmost) gives reward -1; reaching state G (rightmost) gives reward +1. All other transitions give reward 0.

```
  [-1] A - B - C - D - E - F - G [+1]
               start^
```

The **true values** under the random policy are linear: $V(A)=-1, V(B)=-2/3, V(C)=-1/3, V(D)=0, V(E)=1/3, V(F)=2/3, V(G)=+1$. (Terminals excluded from prediction.)

In [ ]:
# Purpose: Define the Random Walk environment
# Key Concept: A minimal MDP with known true values for benchmarking

class RandomWalk:
    """
    7-state random walk. States 1-5 are non-terminal (A-E in the book,
    here indexed 1..5). State 0 and 6 are terminal.
    Reward: -1 at left terminal, +1 at right terminal, 0 otherwise.
    """

    N_STATES = 7          # including 2 terminal states
    START = 3             # center state (0-indexed)
    LEFT_TERMINAL = 0
    RIGHT_TERMINAL = 6

    # True values for non-terminal states (analytically derived)
    TRUE_VALUES = np.array([0, -2/3, -1/3, 0, 1/3, 2/3, 0])
    # Note: terminals have value 0 (absorbing), non-terminals linearly spaced
    # Correct analytical solution with reward ±1 at terminals:
    TRUE_VALUES = np.linspace(-1, 1, 7)  # [-1, -2/3, -1/3, 0, 1/3, 2/3, 1]

    def reset(self):
        """Return start state."""
        return self.START

    def step(self, state):
        """
        Take a random step left or right.

        Returns
        -------
        next_state : int
        reward     : float
        done       : bool
        """
        # Step left or right with equal probability
        next_state = state + np.random.choice([-1, 1])

        if next_state == self.LEFT_TERMINAL:
            return next_state, -1.0, True
        elif next_state == self.RIGHT_TERMINAL:
            return next_state, 1.0, True
        else:
            return next_state, 0.0, False

env = RandomWalk()
print("True values:", env.TRUE_VALUES)
print("States (incl. terminals):", list(range(env.N_STATES)))
print("Non-terminal states:      ", list(range(1, env.N_STATES - 1)))

## 2. TD(0) Prediction Algorithm

TD(0) is the simplest TD method. For each time step within an episode:

1. Observe current state $S_t$
2. Take action (here: random), observe $R_{t+1}$ and $S_{t+1}$
3. Compute TD error: $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$
4. Update: $V(S_t) \leftarrow V(S_t) + \alpha \delta_t$

Notice the update happens **immediately after each transition**, not after the episode ends. This is TD(0)'s key advantage over MC.

In [ ]:
# Purpose: Implement TD(0) prediction
# Key Concept: Online, incremental updates using one-step bootstrapping

def td_zero_prediction(env, n_episodes, alpha=0.1, gamma=1.0, seed=42):
    """
    TD(0) prediction for a given environment with a random policy.

    Parameters
    ----------
    env       : RandomWalk
    n_episodes: int   — number of episodes to run
    alpha     : float — step-size (learning rate)
    gamma     : float — discount factor
    seed      : int   — RNG seed for reproducibility

    Returns
    -------
    V         : np.ndarray, shape (N_STATES,) — learned value estimates
    rms_history : list — RMS error per episode (non-terminal states only)
    """
    rng = np.random.default_rng(seed)

    # Step 1: Initialize value function to 0 for all states
    V = np.zeros(env.N_STATES)

    rms_history = []
    non_terminal = np.arange(1, env.N_STATES - 1)  # states 1..5

    for episode in range(n_episodes):
        state = env.reset()
        done = False

        while not done:
            # Step 2: Take a step (random policy)
            next_state, reward, done = env.step(state)

            # Step 3: Compute TD error
            # V[next_state] = 0 when next_state is terminal (absorbing state)
            td_error = reward + gamma * V[next_state] - V[state]

            # Step 4: Update value estimate
            V[state] += alpha * td_error

            state = next_state

        # Track RMS error over non-terminal states
        rms = np.sqrt(np.mean((V[non_terminal] - env.TRUE_VALUES[non_terminal]) ** 2))
        rms_history.append(rms)

    return V, rms_history


# Run TD(0) for 300 episodes
V_td, rms_td = td_zero_prediction(env, n_episodes=300, alpha=0.1)
print("Learned values (TD(0)):  ", np.round(V_td, 3))
print("True values:             ", np.round(env.TRUE_VALUES, 3))
print(f"Final RMS error: {rms_td[-1]:.4f}")

## 3. Monte Carlo Prediction (Baseline)

To compare fairly, we implement every-visit Monte Carlo prediction. MC must wait until the episode ends, then back-propagates returns. This makes it unbiased but higher-variance than TD.

In [ ]:
# Purpose: Implement every-visit Monte Carlo prediction for comparison
# Key Concept: MC uses the full return G_t; TD uses the one-step bootstrap

def mc_prediction(env, n_episodes, alpha=0.1, gamma=1.0, seed=42):
    """
    Every-visit Monte Carlo prediction with constant step-size alpha.

    Parameters
    ----------
    env       : RandomWalk
    n_episodes: int   — number of episodes
    alpha     : float — constant step-size
    gamma     : float — discount factor

    Returns
    -------
    V           : np.ndarray — learned value estimates
    rms_history : list — RMS error per episode
    """
    rng = np.random.default_rng(seed)
    V = np.zeros(env.N_STATES)
    rms_history = []
    non_terminal = np.arange(1, env.N_STATES - 1)

    for episode in range(n_episodes):
        # Step 1: Generate a full episode trajectory
        trajectory = []   # list of (state, reward) pairs
        state = env.reset()
        done = False

        while not done:
            next_state, reward, done = env.step(state)
            trajectory.append((state, reward))
            state = next_state

        # Step 2: Compute returns and update backward through the episode
        G = 0.0
        for s, r in reversed(trajectory):
            G = r + gamma * G
            V[s] += alpha * (G - V[s])   # incremental mean update

        rms = np.sqrt(np.mean((V[non_terminal] - env.TRUE_VALUES[non_terminal]) ** 2))
        rms_history.append(rms)

    return V, rms_history


V_mc, rms_mc = mc_prediction(env, n_episodes=300, alpha=0.1)
print("Learned values (MC):     ", np.round(V_mc, 3))
print("True values:             ", np.round(env.TRUE_VALUES, 3))
print(f"Final RMS error: {rms_mc[-1]:.4f}")

## 4. Visualization: Learned Values and Convergence

We now compare learned values against the true values and plot how quickly each method converges.

In [ ]:
# Purpose: Visualize learned value estimates and convergence curves
# Key Concept: TD converges faster than MC in terms of episodes on this task

state_labels = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
non_term_idx = list(range(1, 6))
non_term_labels = [state_labels[i] for i in non_term_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left plot: Learned values after 300 episodes ---
ax = axes[0]
ax.plot(non_term_labels, env.TRUE_VALUES[non_term_idx],
        'k--', linewidth=2, label='True values', zorder=5)
ax.plot(non_term_labels, V_td[non_term_idx],
        'o-', color='steelblue', linewidth=2, markersize=8, label='TD(0)')
ax.plot(non_term_labels, V_mc[non_term_idx],
        's-', color='tomato', linewidth=2, markersize=8, label='MC')
ax.set_xlabel('State', fontsize=12)
ax.set_ylabel('Estimated Value', fontsize=12)
ax.set_title('Learned vs. True Values (300 episodes, alpha=0.1)', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(-1.1, 1.1)

# --- Right plot: RMS error over episodes ---
ax = axes[1]
ax.plot(rms_td, color='steelblue', linewidth=1.5, label='TD(0)')
ax.plot(rms_mc, color='tomato', linewidth=1.5, label='MC')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('RMS Error (non-terminal states)', fontsize=12)
ax.set_title('Convergence: TD(0) vs. MC', fontsize=13)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../resources/td_zero_vs_mc.png', dpi=120, bbox_inches='tight')
plt.show()
print("TD(0) final RMS:", round(rms_td[-1], 4), "  MC final RMS:", round(rms_mc[-1], 4))

## 5. Effect of Learning Rate alpha

The step-size alpha controls the speed-accuracy trade-off. Large alpha converges fast but oscillates; small alpha converges slowly but smoothly. Let's run TD(0) across several alpha values and observe.

In [ ]:
# Purpose: Show how alpha affects TD(0) convergence behavior
# Key Concept: alpha is the most important hyperparameter in TD methods

alphas = [0.01, 0.05, 0.1, 0.3, 0.5]
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(alphas)))

fig, ax = plt.subplots(figsize=(10, 5))

for alpha, color in zip(alphas, colors):
    _, rms = td_zero_prediction(env, n_episodes=500, alpha=alpha, seed=0)
    ax.plot(rms, color=color, linewidth=1.5, label=f'alpha={alpha}')

ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('RMS Error', fontsize=12)
ax.set_title('TD(0) Convergence for Different Learning Rates', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### Exercise 3.1.1: Averaging Over Multiple Runs

**Task:** The single-run convergence curves above are noisy. Compute the average RMS error over 20 independent runs for both TD(0) and MC (alpha=0.1, 300 episodes each). Plot the smoothed curves with a shaded region showing ±1 standard deviation across runs.

**Expected Output:** Two smooth curves with shaded confidence bands showing TD(0) converges more reliably than MC.

<details>
<summary>Hint 1</summary>
Run each method 20 times with different seeds. Collect rms_history from each run into a 2D array of shape (n_runs, n_episodes). Then use np.mean and np.std along axis=0.
</details>

<details>
<summary>Hint 2</summary>
Use ax.fill_between(x, mean - std, mean + std, alpha=0.2) to draw the shaded confidence band.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
# Run TD(0) and MC 20 times each, collect RMS histories, plot averaged curves

N_RUNS = 20
N_EPISODES = 300
ALPHA = 0.1

# Collect results: shape (N_RUNS, N_EPISODES)
td_rms_all = np.zeros((N_RUNS, N_EPISODES))
mc_rms_all = np.zeros((N_RUNS, N_EPISODES))

for run in range(N_RUNS):
    _, td_rms_all[run] = td_zero_prediction(env, N_EPISODES, alpha=ALPHA, seed=run)
    _, mc_rms_all[run] = mc_prediction(env, N_EPISODES, alpha=ALPHA, seed=run)

td_mean = td_rms_all.mean(axis=0)
td_std  = td_rms_all.std(axis=0)
mc_mean = mc_rms_all.mean(axis=0)
mc_std  = mc_rms_all.std(axis=0)

episodes = np.arange(N_EPISODES)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(episodes, td_mean, color='steelblue', linewidth=2, label='TD(0) mean')
ax.fill_between(episodes, td_mean - td_std, td_mean + td_std,
                color='steelblue', alpha=0.2, label='TD(0) ±1 std')
ax.plot(episodes, mc_mean, color='tomato', linewidth=2, label='MC mean')
ax.fill_between(episodes, mc_mean - mc_std, mc_mean + mc_std,
                color='tomato', alpha=0.2, label='MC ±1 std')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('RMS Error', fontsize=12)
ax.set_title(f'TD(0) vs MC — Average over {N_RUNS} runs (alpha={ALPHA})', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_311():
    assert td_rms_all.shape == (N_RUNS, N_EPISODES), \
        f"td_rms_all should have shape ({N_RUNS}, {N_EPISODES}), got {td_rms_all.shape}"
    assert mc_rms_all.shape == (N_RUNS, N_EPISODES), \
        f"mc_rms_all should have shape ({N_RUNS}, {N_EPISODES}), got {mc_rms_all.shape}"
    assert td_mean[-1] < 0.25, \
        f"TD(0) final mean RMS={td_mean[-1]:.3f} seems too high — check your alpha or n_episodes"
    assert mc_mean[-1] < 0.30, \
        f"MC final mean RMS={mc_mean[-1]:.3f} seems too high"
    print("Exercise 3.1.1 passed!")

test_exercise_311()

### Exercise 3.1.2: TD Error Magnitude Over Time

**Task:** Modify `td_zero_prediction` to also return the mean absolute TD error per episode. Plot this alongside the RMS error for 300 episodes. Do both quantities decrease together?

<details>
<summary>Hint 1</summary>
Inside the while loop, accumulate abs(td_error) values. At episode end, compute the mean and append to a list.
</details>

<details>
<summary>Hint 2</summary>
Use a secondary y-axis (ax.twinx()) to plot the TD error on a different scale.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def td_zero_with_td_error(env, n_episodes, alpha=0.1, gamma=1.0, seed=42):
    """
    TD(0) prediction that also tracks the mean absolute TD error per episode.

    Returns
    -------
    V           : np.ndarray
    rms_history : list
    td_err_history : list — mean |td_error| per episode
    """
    rng = np.random.default_rng(seed)
    V = np.zeros(env.N_STATES)
    rms_history = []
    td_err_history = []
    non_terminal = np.arange(1, env.N_STATES - 1)

    for _ in range(n_episodes):
        state = env.reset()
        done = False
        episode_td_errors = []

        while not done:
            next_state, reward, done = env.step(state)
            td_error = reward + gamma * V[next_state] - V[state]
            episode_td_errors.append(abs(td_error))
            V[state] += alpha * td_error
            state = next_state

        rms = np.sqrt(np.mean((V[non_terminal] - env.TRUE_VALUES[non_terminal]) ** 2))
        rms_history.append(rms)
        td_err_history.append(np.mean(episode_td_errors))

    return V, rms_history, td_err_history


_, rms_hist, td_err_hist = td_zero_with_td_error(env, n_episodes=300, alpha=0.1)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(rms_hist, color='steelblue', linewidth=1.5, label='RMS Error')
ax2.plot(td_err_hist, color='darkorange', linewidth=1.5, alpha=0.7, label='Mean |TD Error|')

ax1.set_xlabel('Episode', fontsize=12)
ax1.set_ylabel('RMS Error', color='steelblue', fontsize=12)
ax2.set_ylabel('Mean |TD Error|', color='darkorange', fontsize=12)
ax1.set_title('RMS Error and Mean |TD Error| During TD(0) Training', fontsize=13)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# AUTO-GRADED TESTS — Do not modify
# ----------------------------------
def test_exercise_312():
    assert len(td_err_hist) == 300, \
        f"td_err_history should have 300 entries, got {len(td_err_hist)}"
    assert all(e >= 0 for e in td_err_hist), \
        "All TD errors should be non-negative (we track absolute values)"
    # TD errors should generally decrease over training
    assert np.mean(td_err_hist[:20]) > np.mean(td_err_hist[-20:]), \
        "Mean |TD error| should be higher early in training than at the end"
    print("Exercise 3.1.2 passed!")

test_exercise_312()

## Key Takeaways

1. **TD(0) bootstraps**: it updates value estimates using other estimates (not just rewards), enabling online learning within episodes.
2. **The TD error** $\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$ is the workhorse of modern RL — it appears in every algorithm from Q-learning to PPO.
3. **TD converges faster than MC** on most tasks because it uses information from every step, not just episode endings.
4. **Alpha is critical**: too large causes oscillation; too small causes slow convergence. Values in [0.05, 0.3] are typical starting points.
5. **Bias vs variance**: TD has lower variance but is biased (bootstraps from imperfect estimates). MC is unbiased but high variance. Both converge to $V_\pi$ given enough data.

## What's Next

Notebook 2 extends TD ideas to **control** (finding the optimal policy) via SARSA and Q-learning. You will see how the same TD error drives policy improvement, and observe the famous behavioral difference between on-policy and off-policy methods on the Cliff Walking problem.

**Additional Resources**
- Sutton & Barto, Chapter 6 (Temporal-Difference Learning)
- David Silver's RL Lecture 4: Model-Free Prediction